In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/post_cell_7.pickle

In [ ]:
%%cudf.pandas.profile
### cell 8 ###

# Optimized Cell 8
cols = ['Avg. Avg D Kbps', 'Avg. Avg U Kbps', 'Avg Lat Ms']

def compute_stats(df):
    # one GPU-accelerated groupby to pull out first/last of each metric
    agg = df.groupby('Name').agg(
        first_download = ('Avg. Avg D Kbps', 'first'),
        last_download  = ('Avg. Avg D Kbps', 'last'),
        first_upload   = ('Avg. Avg U Kbps', 'first'),
        last_upload    = ('Avg. Avg U Kbps', 'last'),
        first_latency  = ('Avg Lat Ms',       'first'),
        last_latency   = ('Avg Lat Ms',       'last'),
    )
    # compute deltas directly on the GPU and select only the three change columns
    stats = agg.assign(
        Change_Download = agg.last_download  - agg.first_download,
        Change_Upload   = agg.last_upload    - agg.first_upload,
        Change_Latency  = agg.last_latency   - agg.first_latency,
    )[[ 'Change_Download', 'Change_Upload', 'Change_Latency' ]]
    return stats

Mobile_Stats    = compute_stats(Mobile_df)
Broadband_Stats = compute_stats(Broadband_df)

# assemble final table of download deltas only
Total_Stats = (
    Mobile_Stats[['Change_Download']]
      .rename(columns={'Change_Download':'Mobile'})
    .join(
      Broadband_Stats[['Change_Download']]
        .rename(columns={'Change_Download':'Broadband'})
    )
)
